In [15]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig

Defining the model

In [16]:
quant_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer=AutoTokenizer.from_pretrained(model_name) #Text → Tokens

model=AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", quantization_config=quant_config ) #Downloads + loads weights

Loading weights: 100%|██████████| 201/201 [00:10<00:00, 18.45it/s]


Testing the model

In [32]:
prompt = "### Instruction: What is the capital of Himachal Pradesh is\n### Response:"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print(inputs)

print("////////////////////////////////////////////////////////")

outputs = model.generate(**inputs, max_new_tokens=13)

print(outputs)

Both `max_new_tokens` (=13) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'input_ids': tensor([[    1,   835,  2799,  4080, 29901,  1724,   338,   278,  7483,   310,
         18136,   496,   284,  1588, 21754,   338,    13,  2277, 29937, 13291,
         29901]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
////////////////////////////////////////////////////////
tensor([[    1,   835,  2799,  4080, 29901,  1724,   338,   278,  7483,   310,
         18136,   496,   284,  1588, 21754,   338,    13,  2277, 29937, 13291,
         29901,   450,  7483,   310, 18136,   496,   284,  1588, 21754,   338,
          1383,   326,   433, 29889]])


autoregressive generation

Input tokens →
Model predicts next token →
Adds it →
Predicts next →

In [33]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction: What is the capital of Himachal Pradesh is
### Response: The capital of Himachal Pradesh is Shimla.


LLMs use self-attention to understand context, then generate tokens one by one

Preparing the dataset

In [35]:
data = [
    {
        "text": "### Instruction: I feel very anxious about my future\n### Response: It sounds like you're feeling overwhelmed about what lies ahead. That’s a very human experience. Instead of trying to push that feeling away, we can acknowledge it and still focus on what matters to you right now. What is one small step you can take today?"
    },
    {
        "text": "### Instruction: I feel like I am not good enough\n### Response: I'm really sorry you're feeling this way. Sometimes our minds create thoughts that feel very real, but they are not always facts. You are more than these thoughts. What is something important to you that you would still like to move toward?"
    },
    {
        "text": "### Instruction: I am stressed about exams\n### Response: It makes sense to feel stressed in this situation. Instead of fighting the stress, we can accept that it’s there and still take action. What is one small thing you can do right now to prepare?"
    }
]

dataset = Dataset.from_list(data)

Tokenizing the dataset

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],  # take the text from dataset (Instruction + Response)
        truncation=True,  ## cut text if it exceeds max_length (prevents overflow)
        padding="max_length", ##adding zeroes to make all sequences the same length (max_length)
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize)

Map: 100%|██████████| 3/3 [00:00<00:00, 23.00 examples/s]


After dataset.map(tokenize)

Your dataset becomes:

[
  {
    "input_ids": [...256 tokens...],
    "attention_mask": [...256 values...]
  }
]

configuration of LORA

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], # attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


Q (Query) → what to focus on  
K (Key)   → what is available  
V (Value) → actual information

Added labels

input_ids:	what model reads

labels:	    what model should output

loss:	    difference between them

Input:   [I,     feel,   stressed]

Label:        [feel,   stressed,  next]

In [ ]:
def add_labels(example):
    example["labels"] = example["input_ids"]  #token → predict next token
    return example

tokenized_dataset = tokenized_dataset.map(add_labels)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map: 100%|██████████| 3/3 [00:00<00:00, 45.40 examples/s]


Now each row becomes:
{
  
  "input_ids": [835, 2799, 310, ...],

  "attention_mask": [1,1,1,...],
  
  "labels": [835, 2799, 310, ...]
}

In [ ]:

training_args = TrainingArguments(
    output_dir="./lora-act",
    per_device_train_batch_size=2,
    num_train_epochs=5,
    learning_rate=2e-4, #Controls how fast model updates weights
    logging_steps=10, #Print training logs every 10 steps
    save_steps=50 #Save model checkpoint every 50 steps
)

In [41]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

Finall training the model

In [42]:
trainer.train()

Step,Training Loss
10,12.476770


TrainOutput(global_step=10, training_loss=12.47677001953125, metrics={'train_runtime': 19806.019, 'train_samples_per_second': 0.001, 'train_steps_per_second': 0.001, 'total_flos': 23861117583360.0, 'train_loss': 12.47677001953125, 'epoch': 5.0})

saving_dir = "./lora-act-final"

In [43]:
model.save_pretrained("act-lora-final")
tokenizer.save_pretrained("act-lora-final")

('act-lora-final/tokenizer_config.json',
 'act-lora-final/chat_template.jinja',
 'act-lora-final/tokenizer.json')

In [44]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(model_name)

# Load LoRA model
model = PeftModel.from_pretrained(base_model, "act-lora-final")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("act-lora-final")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1458.04it/s]


In [45]:
prompt = "### Instruction: I feel stressed about exams\n### Response:"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction: I feel stressed about exams
### Response: I’m in the same boat, I feel the same way. I usually feel more stressed leading up to exams, but I try to prepare as much as possible. I also find it helpful to take some deep breaths and meditate during the day to calm my nerves. I also try to enjoy my free time and make sure to take care of myself, eating healthy and getting enough sleep. I’ve also found that talking to a trusted friend or therap
